In [15]:
import torch
from torch import nn
import torchvision
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision.io import decode_image
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import os
import numpy as np

In [16]:
writer = SummaryWriter()

In [17]:
class ISIC2019(Dataset): # TODO: consider maybe removing downsampled or removing duplicates. unsure if these are necessary
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.ohe_labels = self.img_labels.iloc[:, 1:]
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0], ".jpg")
        image = decode_image(img_path)
        label = np.where(self.ohe_labels==1)[1][idx]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

In [18]:
isic_dataset = ISIC2019(img_dir='../data/ISIC_2019_Training_Input', annotations_file='../data/ISIC_2019_Training_GroundTruth.csv')
train_images_size = len(isic_dataset)

In [ ]:
train_size = int(train_images_size * 0.80)
test_size = train_images_size - train_size
# 80% train 20% test

isic_train, isic_test = random_split(isic_dataset, [train_size, test_size])
train_size, test_size

(20264, 5067, 20264)

In [ ]:
class MobileNetV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.model =  nn.Sequential(nn.Conv2d(1,6,5, padding=2),
                                    nn.Sigmoid(),
                                    nn.AvgPool2d(2, stride=2),
                                    nn.Conv2d(6,16,5),
                                    nn.Sigmoid(),
                                    nn.AvgPool2d(2, stride=2),
                                    nn.Flatten(),
                                    nn.Linear(400, 120),
                                    nn.Sigmoid(),
                                    nn.Linear(120, 84),
                                    nn.Sigmoid(),
                                    nn.Linear(84, 10))

    def forward(self, x):
        x = self.model(x)
        return x

In [ ]:
mobilenet = MobileNetV3()
mobilenet

In [ ]:
epochs = 100
learning_rate = 1e-4
batch_size = 16

In [ ]:
dataloader_train = DataLoader(isic_train, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(isic_test, batch_size=batch_size, shuffle=True)

num_train_batches = len(dataloader_train)
num_test_batches = len(dataloader_test)

loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mobilenet.parameters(), lr=learning_rate)

In [ ]:
for epoch in range(epochs):
    train_loss = 0
    train_acc = 0

    mobilenet.train()
    for batch_idx, (train_features, train_labels) in enumerate(dataloader_train):
        
        optimizer.zero_grad()

        predictions = mobilenet(train_features)
        predictions_labels = torch.argmax(predictions, dim=1)

        train_batch_acc = (predictions_labels == train_labels).sum().item() / train_features.shape[0]

        train_batch_loss = loss(predictions, train_labels)
        train_batch_loss.backward()

        optimizer.step()

        train_loss += train_batch_loss.item()
        train_acc += train_batch_acc

    val_loss = 0
    val_acc = 0

    mobilenet.eval()
    with torch.no_grad():
        for batch_idx, (test_features, test_labels) in enumerate(dataloader_test):
            predictions = mobilenet(test_features)
            predictions_labels = torch.argmax(predictions, dim=1)

            test_batch_acc = (predictions_labels == test_labels).sum().item() / test_features.shape[0]
            test_batch_loss = loss(predictions, test_labels)

            val_loss += test_batch_loss.item()
            val_acc += test_batch_acc

    train_loss /= num_train_batches
    train_acc /= num_train_batches

    val_loss /= num_test_batches
    val_acc /= num_test_batches

    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_acc, epoch)

    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar('Accuracy/val', val_acc, epoch)

In [100]:
writer.flush()